# Finish InferenceLab qualification 00B

Copy the single code cell into your **existing Kaggle notebook/session** and run it. Do not start a fresh GPU session: this batch reuses `/kaggle/working/qualification/kaggle-t4-20261025-b/.venv` and attempt c artifacts.

The cell stops the baseline server, checks torch.compile and minimal Triton/RMSNorm correctness, then starts/stops TP=2 and AWQ servers sequentially. AWQ is the explicitly selected `Qwen/Qwen2.5-3B-Instruct-AWQ`; its immutable revision, publisher source, architecture, vocabulary and template are verified before loading. No package installation, dtype/backend change, performance claim, or platform implementation occurs. Each failure is preserved and later independent checks continue when GPUs are idle.

Download the ZIP printed at the end **before ending the session**. It includes attempts b/c, manifests, logs and scripts, excluding environments and bootstrap binaries. Return the final status table and ZIP to the chat.

Only local syntax and offline self-checks have been run during preparation. GPU outcomes are not known until you execute this cell.


In [ ]:
from pathlib import Path
import subprocess
import sys

script = Path('/kaggle/working/finish_00b.py')
script.write_text('"""Finish prompt 00B in the existing Kaggle session; never install packages."""\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shlex\nimport shutil\nimport signal\nimport socket\nimport subprocess\nimport sys\nimport tempfile\nimport time\nfrom datetime import datetime, timezone\nfrom importlib.metadata import version\nfrom urllib.error import URLError\nfrom urllib.request import urlopen\nimport zipfile\n\nREPO = "Qwen/Qwen2.5-3B-Instruct"\nREVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"\nCAPABILITIES = ("single_gpu_fp16_vllm", "streaming", "torch_compile", "triton", "tp2", "awq")\n\n\ndef read_json(path):\n    return json.loads(path.read_text())\n\n\ndef write_json(path, value):\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n\n\ndef launch_command(path):\n    parts = shlex.split([s for s in path.read_text().splitlines() if s.strip()][-1])\n    settings = {}\n    while parts and re.match(r"^[A-Z_][A-Z_0-9]*=", parts[0]):\n        key, value = parts.pop(0).split("=", 1)\n        settings[key] = value\n    assert settings == {"CUDA_VISIBLE_DEVICES": "0", "VLLM_USE_V1": "0", "VLLM_ATTENTION_BACKEND": "XFORMERS"}, settings\n    expected = {"--model": REPO, "--revision": REVISION, "--tokenizer-revision": REVISION,\n                "--dtype": "float16", "--tensor-parallel-size": "1", "--port": "8001"}\n    for flag, value in expected.items():\n        assert parts[parts.index(flag) + 1] == value, (flag, value)\n    assert "--no-enable-chunked-prefill" in parts\n    assert "--no-enable-prefix-caching" in parts\n    return settings, parts\n\n\ndef check_stream(result):\n    assert result["status"] == "VALIDATED" and result["done_received"]\n    assert result["finish_reason"] == "length"\n    assert result["token_count_source"] == "server_usage"\n    assert result["usage"]["completion_tokens"] == 128\n    assert result["usage"]["prompt_tokens"] > 0\n    start, first, last, end = (result[k] for k in (\n        "request_start_ns", "first_content_ns", "last_content_ns", "terminal_ns"))\n    assert all(type(v) is int for v in (start, first, last, end))\n    assert start <= first <= last <= end and result["content_events"] >= 2\n    expected = {"client_ttft_ms": (first-start)/1e6, "client_e2e_ms": (end-start)/1e6,\n                "client_tpot_estimate_ms": (last-first)/1e6/127}\n    for metric, value in expected.items():\n        assert math.isclose(result["metrics"][metric]["value"], value, rel_tol=1e-9, abs_tol=1e-9)\n\n\nGPU_TEST = r\'\'\'\nimport sys\nimport torch\nmode = sys.argv[1]\ntorch.manual_seed(42)\nassert torch.cuda.get_device_capability(0) == (7, 5)\ndef reference(x, w):\n    z = x.float()\n    return (z * torch.rsqrt((z*z).mean(-1, keepdim=True) + 1e-6) * w.float()).to(x.dtype)\n\nif mode == "torch_compile":\n    torch._dynamo.config.suppress_errors = False\n    compiled = torch.compile(reference, backend="inductor", fullgraph=True)\n    for seed in (7, 42):\n        torch.manual_seed(seed)\n        for shape in ((4, 128), (16, 2048)):\n            x = torch.randn(shape, device="cuda", dtype=torch.float16)\n            w = torch.randn(shape[-1], device="cuda", dtype=torch.float16)\n            # Correctness before compilation, then compiled and post-compilation execution.\n            torch.testing.assert_close(reference(x, w), torch.nn.functional.rms_norm(\n                x.float(), (shape[-1],), w.float(), eps=1e-6).half(), rtol=2e-3, atol=2e-3)\n            for _ in range(2):\n                y = compiled(x, w)\n                torch.cuda.synchronize()\n                torch.testing.assert_close(y, reference(x, w), rtol=2e-3, atol=2e-3)\nelse:\n    import triton\n    import triton.language as tl\n    @triton.jit\n    def add(X, Y, Z, N: tl.constexpr, B: tl.constexpr):\n        i = tl.program_id(0) * B + tl.arange(0, B)\n        tl.store(Z+i, tl.load(X+i, i<N, 0)+tl.load(Y+i, i<N, 0), i<N)\n    @triton.jit\n    def rms(X, W, Y, N: tl.constexpr, B: tl.constexpr):\n        row = tl.program_id(0)\n        i = tl.arange(0, B)\n        x = tl.load(X+row*N+i, i<N, 0).to(tl.float32)\n        w = tl.load(W+i, i<N, 0).to(tl.float32)\n        y = x * tl.rsqrt(tl.sum(x*x, 0)/N + 1e-6) * w\n        tl.store(Y+row*N+i, y, i<N)\n    if mode == "triton_minimal":\n        x = torch.randn(1025, device="cuda", dtype=torch.float16)\n        y = torch.randn_like(x)\n        z = torch.empty_like(x)\n        add[(triton.cdiv(x.numel(), 128),)](x, y, z, x.numel(), 128, num_warps=4, num_stages=1)\n        torch.cuda.synchronize()\n        torch.testing.assert_close(z, x+y, rtol=0, atol=0)\n    elif mode == "triton_rmsnorm":\n        for seed in (7, 42):\n            torch.manual_seed(seed)\n            for shape in ((4, 128), (16, 2048)):\n                x = torch.randn(shape, device="cuda", dtype=torch.float16)\n                w = torch.randn(shape[-1], device="cuda", dtype=torch.float16)\n                y = torch.empty_like(x)\n                rms[(shape[0],)](x, w, y, shape[-1], triton.next_power_of_2(shape[-1]), num_warps=4, num_stages=1)\n                torch.cuda.synchronize()\n                torch.testing.assert_close(y, reference(x, w), rtol=2e-3, atol=2e-3)\n    else:\n        raise ValueError(mode)\nprint(mode + ": VALIDATED; correctness only, no performance measurement", flush=True)\n\'\'\'\n\n\ndef run(command, path, timeout=60, env=None):\n    with path.open("a") as out:\n        out.write("$ " + shlex.join(map(str, command)) + "\\n")\n        out.flush()\n        process = subprocess.Popen(command, stdout=out, stderr=subprocess.STDOUT, env=env, start_new_session=True)\n        try:\n            code = process.wait(timeout=timeout)\n        except BaseException:\n            stop_group(process.pid)\n            process.wait(timeout=15)\n            raise\n    if code:\n        raise RuntimeError(f"Exit {code}; see {path}")\n\n\ndef gpu_idle():\n    result = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader,nounits"],\n                            capture_output=True, text=True, timeout=10)\n    if result.returncode or result.stdout.strip():\n        raise BlockingIOError("GPU compute processes remain or cannot be inspected: " + result.stdout + result.stderr)\n\n\ndef stop_group(pid):\n    def identity(pid):\n        try:\n            fields = Path(f"/proc/{pid}/stat").read_text().rsplit(") ", 1)[1].split()\n            return int(fields[1]), int(fields[19])\n        except (OSError, ValueError, IndexError):\n            return None\n\n    # vLLM workers may create new sessions. Snapshot descendants before stopping\n    # their parent; start times prevent signalling a subsequently reused PID.\n    processes = {int(p.name): identity(int(p.name)) for p in Path("/proc").glob("[0-9]*")}\n    owned = {pid} if processes.get(pid) else set()\n    while children := {p for p, info in processes.items() if info and info[0] in owned} - owned:\n        owned.update(children)\n    for sig in (signal.SIGTERM, signal.SIGKILL):\n        for child in owned:\n            current = identity(child)\n            if current and current[1] == processes[child][1]:\n                try:\n                    os.kill(child, sig)\n                except ProcessLookupError:\n                    pass\n        if sig == signal.SIGTERM and owned:\n            time.sleep(3)\n\n\ndef stop_baseline(q):\n    pid = int((q / "server-pid.txt").read_text())\n    process = Path(f"/proc/{pid}/cmdline")\n    if process.exists() and (args := process.read_bytes().split(b"\\0")) != [b""]:\n        assert b"vllm.entrypoints.openai.api_server" in args, "PID no longer belongs to our server"\n        assert REPO.encode() in args and REVISION.encode() in args\n        assert os.getpgid(pid) == pid, "Unexpected process group; refusing to kill"\n        stop_group(pid)\n    gpu_idle()\n\n\ndef server_smoke(stage, command, settings, probe, expected):\n    gpu_idle()\n    with socket.socket() as sock:\n        sock.bind(("127.0.0.1", 8001))\n    write_json(stage / "launch-settings.json", {"environment": settings, "expected": expected})\n    (stage / "server-command.txt").write_text(shlex.join([f"{k}={v}" for k, v in settings.items()] + command) + "\\n")\n    env = dict(os.environ, **settings)\n    with (stage / "server-log.txt").open("w") as log:\n        process = subprocess.Popen(command, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n    try:\n        deadline = time.monotonic() + 900\n        update = time.monotonic() + 30\n        while True:\n            if process.poll() is not None:\n                raise RuntimeError(f"Server exited {process.returncode}; see {stage / \'server-log.txt\'}")\n            try:\n                with urlopen("http://127.0.0.1:8001/health", timeout=2) as response:\n                    if response.status == 200:\n                        break\n            except (URLError, TimeoutError):\n                pass\n            if time.monotonic() >= deadline:\n                raise TimeoutError("Server startup exceeded 15 minutes")\n            if time.monotonic() >= update:\n                print(stage.name + ": still starting", flush=True)\n                update = time.monotonic() + 30\n            time.sleep(2)\n        # ponytail: these checks are specific to vLLM 0.10.2 startup text;\n        # use structured runtime introspection if the runtime/log format changes.\n        text = (stage / "server-log.txt").read_text(errors="replace")\n        for marker in ("Using XFormers backend", "dtype=torch.float16", "Initializing a V0 LLM engine",\n                       "chunked_prefill_enabled=False", "enable_prefix_caching=False",\n                       f"tensor_parallel_size={expected[\'tp\']}", f"revision={expected[\'revision\']}",\n                       f"quantization={expected[\'quantization\']}"):\n            assert marker in text, "Effective runtime value not verified: " + marker\n        write_json(stage / "runtime-manifest.json", {"effective_checked_values": expected,\n                    "evidence": "server-log.txt", "visible_devices": settings["CUDA_VISIBLE_DEVICES"]})\n        run([sys.executable, "-u", str(probe), str(stage)], stage / "probe.log", 240, env)\n        check_stream(json.loads((stage / "stream-results.jsonl").read_text().splitlines()[-1]))\n    finally:\n        stop_group(process.pid)\n        process.wait(timeout=15)\n\n\ndef self_check():\n    result = {"status": "VALIDATED", "done_received": True, "finish_reason": "length",\n              "token_count_source": "server_usage", "usage": {"completion_tokens": 128, "prompt_tokens": 41},\n              "request_start_ns": 100, "first_content_ns": 1000100,\n              "last_content_ns": 128000100, "terminal_ns": 129000100, "content_events": 2,\n              "metrics": {"client_ttft_ms": {"value": 1}, "client_e2e_ms": {"value": 129},\n                          "client_tpot_estimate_ms": {"value": 1}}}\n    check_stream(result)\n    for key, value in (("done_received", False), ("content_events", 1), ("first_content_ns", None)):\n        bad = dict(result, **{key: value})\n        try:\n            check_stream(bad)\n        except (AssertionError, TypeError):\n            continue\n        raise AssertionError("Invalid stream accepted: " + key)\n    with tempfile.TemporaryDirectory() as tmp:\n        path = Path(tmp) / "server-command.txt"\n        command = ("CUDA_VISIBLE_DEVICES=0 VLLM_USE_V1=0 VLLM_ATTENTION_BACKEND=XFORMERS /example/python "\n                   f"--model {REPO} --revision {REVISION} --tokenizer-revision {REVISION} "\n                   "--dtype float16 --tensor-parallel-size 1 --port 8001 "\n                   "--no-enable-chunked-prefill --no-enable-prefix-caching")\n        path.write_text(command)\n        settings, parsed = launch_command(path)\n        assert settings["VLLM_ATTENTION_BACKEND"] == "XFORMERS" and parsed[0] == "/example/python"\n        path.write_text(command.replace("--dtype float16", "--dtype bfloat16"))\n        try:\n            launch_command(path)\n        except AssertionError:\n            pass\n        else:\n            raise AssertionError("Dtype change was accepted")\n    compile(GPU_TEST, "gpu_checks.py", "exec")\n    with tempfile.TemporaryDirectory() as tmp:\n        pid_path = Path(tmp) / "child.pid"\n        child_code = ("import subprocess,sys,time,pathlib; "\n                      "p=subprocess.Popen([sys.executable,\'-c\',\'import time; time.sleep(60)\'],start_new_session=True); "\n                      "pathlib.Path(sys.argv[1]).write_text(str(p.pid)); time.sleep(60)")\n        parent = subprocess.Popen([sys.executable, "-c", child_code, str(pid_path)], start_new_session=True)\n        child = None\n        try:\n            deadline = time.monotonic() + 5\n            while not pid_path.exists() and time.monotonic() < deadline:\n                time.sleep(0.05)\n            child = int(pid_path.read_text())\n            assert os.getpgid(child) != os.getpgid(parent.pid)\n            stop_group(parent.pid)\n            parent.wait(timeout=5)\n            status = Path(f"/proc/{child}/stat")\n            assert not status.exists() or status.read_text().rsplit(") ", 1)[1].split()[0] == "Z"\n        finally:\n            for target in (parent.pid, child):\n                if target is not None:\n                    try:\n                        os.kill(target, signal.SIGKILL)\n                    except ProcessLookupError:\n                        pass\n            parent.wait(timeout=5)\n    print("Self-check passed: valid metrics accepted; incomplete/insufficient streams rejected.")\n\n\ndef main(q):\n    old = q.with_name("kaggle-t4-20261025-b")\n    python = old / ".venv/bin/python"\n    assert python.exists(), "Existing qualification environment missing; no packages will be installed"\n    if Path(sys.executable).absolute() != python.absolute():\n        os.execv(str(python), [str(python), str(Path(__file__).resolve()), "--directory", str(q)])\n    assert platform.python_version() == "3.12.12"\n    for package, pin in {"torch": "2.8.0", "vllm": "0.10.2", "triton": "3.4.0", "transformers": "4.55.2", "xformers": "0.0.32.post1"}.items():\n        assert version(package).split("+")[0] == pin, (package, version(package))\n    settings, command = launch_command(q / "server-command.txt")\n    stream = json.loads((q / "stream-results.jsonl").read_text().splitlines()[-1])\n    check_stream(stream)\n    pin = read_json(q / "model-pin.json")\n    assert pin["repo_id"] == REPO and pin["revision"] == REVISION\n    log = (q / "server-log.txt").read_text(errors="replace")\n    # ponytail: version-specific evidence markers, not a general runtime-log parser.\n    for marker in ("dtype=torch.float16", "tensor_parallel_size=1", "Using XFormers backend",\n                   "chunked_prefill_enabled=False", "enable_prefix_caching=False", "enforce_eager=True",\n                   "quantization=None", f"revision={REVISION}"):\n        assert marker in log, marker\n    probe = q / "stream-probe.py"\n    assert probe.exists()\n    batch = q / ("additional-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))\n    batch.mkdir()\n    shutil.copy2(__file__, batch / "finish_00b.py")\n    state = {"qualification_id": q.name, **{c: "NOT_TESTED" for c in CAPABILITIES},\n             "reasons": {c: "Not yet executed" for c in CAPABILITIES}, "model": pin,\n             "validated_versions": {}, "evidence": {}, "qualification_complete": False}\n    for name in ("single_gpu_fp16_vllm", "streaming"):\n        state[name] = "VALIDATED"\n        state["reasons"][name] = "Baseline startup log and successful real streaming generation; chunked prefill disabled."\n    state["baseline_metrics"] = stream["metrics"]\n    state["known_failure"] = {"qualification_id": old.name, "status": "FAILED",\n        "scope": "chunked prefill Triton attention kernel", "shared_memory_required_bytes": 98304,\n        "shared_memory_limit_bytes": 65536}\n    if (q / "qualification.json").exists():\n        shutil.copy2(q / "qualification.json", batch / "previous-qualification.json")\n\n    def record(name, status, reason, evidence):\n        state[name] = status\n        state["reasons"][name] = reason\n        state["evidence"][name] = str(evidence.relative_to(q))\n        write_json(q / "qualification.json", state)\n        print(f"{name}: {status} — {reason}", flush=True)\n\n    try:\n        print("Stopping the baseline server before sequential qualification checks.", flush=True)\n        stop_baseline(q)\n        from importlib.metadata import distributions\n        inventory_code = \'\'\'import torch, json, sys\nfrom pathlib import Path\ngpus = [{"index": i, "name": torch.cuda.get_device_name(i),\n         "compute_capability": list(torch.cuda.get_device_capability(i)),\n         "vram_bytes": torch.cuda.get_device_properties(i).total_memory}\n        for i in range(torch.cuda.device_count())]\nPath(sys.argv[1]).write_text(json.dumps({"gpus": gpus, "cuda": torch.version.cuda}))\n\'\'\'\n        run([sys.executable, "-c", inventory_code, str(batch / "cuda-inventory.json")], batch / "inventory.log")\n        inventory = read_json(batch / "cuda-inventory.json")\n        gpus = inventory["gpus"]\n        assert gpus and all("T4" in gpu["name"] and gpu["compute_capability"] == [7, 5] for gpu in gpus), gpus\n        environment = {"qualification_id": q.name, "captured_at_utc": datetime.now(timezone.utc).isoformat(),\n            "python": platform.python_version(), "os": platform.platform(), "cpu_count": os.cpu_count(),\n            "ram": [line.strip() for line in Path("/proc/meminfo").read_text().splitlines() if line.startswith("MemTotal:")],\n            "gpus_visible_to_inventory": gpus, "inventory_cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),\n            "pytorch_cuda_runtime": inventory["cuda"], "versions": {p: version(p) for p in ("torch", "vllm", "triton", "transformers", "xformers")},\n            "model": pin, "baseline_used_gpu_indices": [0], "tp2_used_gpu_indices": None,\n            "gpu_uuid_driver_and_memory_evidence": str((batch / "gpu-inventory.csv").relative_to(q)),\n            "driver_supported_cuda_evidence": "nvidia-smi.txt", "cuda_toolkit": None,\n            "git_sha": None, "git_sha_reason": "Repository is not mounted in the Kaggle notebook; script SHA256 is preserved instead.",\n            "script_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}\n        for args, file in [(["nvidia-smi"], "nvidia-smi.txt"), (["nvidia-smi", "topo", "-m"], "nvidia-smi-topo.txt")]:\n            run(args, q / file)\n        run(["nvidia-smi", "--query-gpu=index,uuid,name,memory.total,driver_version", "--format=csv"], batch / "gpu-inventory.csv")\n        inventory_rows = [line for line in (batch / "gpu-inventory.csv").read_text().splitlines()\n                          if re.match(r"^\\d+,\\s*GPU-", line)]\n        environment["gpu_present_count"] = len(inventory_rows)\n        environment["gpu_visible_count"] = len(gpus)\n        environment["nvidia_driver_version"] = inventory_rows[0].split(",")[-1].strip() if inventory_rows else None\n        cuda_match = re.search(r"CUDA Version:\\s*([\\d.]+)", (q / "nvidia-smi.txt").read_text())\n        environment["driver_supported_cuda"] = cuda_match.group(1) if cuda_match else None\n        if shutil.which("nvcc"):\n            run(["nvcc", "--version"], batch / "nvcc-version.txt")\n            environment["cuda_toolkit"] = str((batch / "nvcc-version.txt").relative_to(q))\n        else:\n            environment["cuda_toolkit_reason"] = "nvcc unavailable; do not confuse driver CUDA capability with installed toolkit."\n        write_json(q / "environment.json", environment)\n        (batch / "pip-freeze.txt").write_text("\\n".join(sorted(f"{d.metadata[\'Name\']}=={d.version}" for d in distributions())) + "\\n")\n        state["validated_versions"] = {"python": platform.python_version(), **environment["versions"]}\n        state["validated_versions_scope"] = "Baseline serving; individual compile/Triton/TP2/AWQ support is reported separately."\n        write_json(q / "runtime-manifest.json", {"evidence": "server-log.txt", "engine": "V0", "attention_backend": "XFORMERS",\n            "dtype": "float16", "tensor_parallel_size": 1, "quantization": None, "enable_chunked_prefill": False,\n            "enable_prefix_caching": False, "enforce_eager": True, "model": pin, "gpu_used_indices": [0],\n            "launch_command_evidence": "server-command.txt", "kv_cache_dtype": "auto", "kv_cache_resolved_dtype": None,\n            "kv_cache_resolved_dtype_reason": "Startup logs record auto; not independently introspected."})\n\n        tests = batch / "gpu_checks.py"\n        tests.write_text(GPU_TEST)\n        for mode, seconds in (("torch_compile", 600), ("triton_minimal", 180), ("triton_rmsnorm", 180)):\n            path = batch / (mode + ".log")\n            try:\n                print(f"Running {mode}; timeout {seconds}s; log: {path}", flush=True)\n                gpu_idle()\n                run([sys.executable, "-u", str(tests), mode], path, seconds,\n                    dict(os.environ, CUDA_VISIBLE_DEVICES="0"))\n                record(mode, "VALIDATED", "Executed on T4 and passed numerical correctness checks.", path)\n            except BlockingIOError as exc:\n                record(mode, "NOT_TESTED", f"GPU prerequisite blocked: {exc}", path)\n            except Exception as exc:\n                record(mode, "FAILED", f"{type(exc).__name__}: {exc}", path)\n        triton_checks = (state["triton_minimal"], state["triton_rmsnorm"])\n        state["triton"] = "VALIDATED" if all(s == "VALIDATED" for s in triton_checks) else ("NOT_TESTED" if "NOT_TESTED" in triton_checks else "FAILED")\n        state["reasons"]["triton"] = "Scoped to minimal add and small RMSNorm; the chunked-prefill attention kernel in attempt b remains FAILED."\n\n        tp = batch / "tp2"\n        tp.mkdir()\n        if len(gpus) < 2:\n            record("tp2", "UNAVAILABLE", "Two visible T4s are required; no hardware fallback.", tp)\n        else:\n            try:\n                print("Starting TP=2 with the same FP16 checkpoint and two T4s.", flush=True)\n                tp_command = command.copy()\n                tp_command[tp_command.index("--tensor-parallel-size") + 1] = "2"\n                tp_command += ["--distributed-executor-backend", "mp"]\n                server_smoke(tp, tp_command, dict(settings, CUDA_VISIBLE_DEVICES="0,1"), probe,\n                             {"tp": 2, "revision": REVISION, "quantization": "None"})\n                environment["tp2_used_gpu_indices"] = [0, 1]\n                record("tp2", "VALIDATED", "Same FP16 checkpoint, two T4s, TP=2, mp executor, real streaming generation; no speedup claim.", tp)\n            except BlockingIOError as exc:\n                record("tp2", "NOT_TESTED", f"GPU prerequisite blocked: {exc}", tp)\n            except Exception as exc:\n                record("tp2", "FAILED", f"{type(exc).__name__}: {exc}", tp)\n\n        awq = batch / "awq"\n        awq.mkdir()\n        from requests.exceptions import RequestException\n        try:\n            print("Verifying and pinning Qwen/Qwen2.5-3B-Instruct-AWQ before loading.", flush=True)\n            gpu_idle()\n            from huggingface_hub import HfApi, hf_hub_download\n            from transformers import AutoTokenizer\n            awq_repo = REPO + "-AWQ"\n            info = HfApi().model_info(awq_repo, timeout=60)\n            assert re.fullmatch(r"[0-9a-f]{40}", info.sha)\n            provenance = info.card_data.to_dict().get("base_model") if info.card_data else None\n            assert provenance == REPO or provenance == [REPO], "Unverified AWQ source model"\n            awq_pin = {"repo_id": awq_repo, "revision": info.sha, "tokenizer_revision": info.sha, "base_model": provenance}\n            write_json(awq / "model-pin.json", awq_pin)\n            config = read_json(Path(hf_hub_download(awq_repo, "config.json", revision=info.sha)))\n            source = read_json(Path(hf_hub_download(REPO, "config.json", revision=REVISION)))\n            quantization = config["quantization_config"]\n            assert quantization["quant_method"] == "awq" and quantization["bits"] == 4\n            for key in ("model_type", "hidden_size", "intermediate_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "vocab_size"):\n                assert config[key] == source[key], "Source model mismatch: " + key\n            baseline_tokenizer = AutoTokenizer.from_pretrained(REPO, revision=REVISION, trust_remote_code=False)\n            quant_tokenizer = AutoTokenizer.from_pretrained(awq_repo, revision=info.sha, trust_remote_code=False)\n            assert baseline_tokenizer.get_vocab() == quant_tokenizer.get_vocab(), "Tokenizer vocabulary differs"\n            assert baseline_tokenizer.chat_template == quant_tokenizer.chat_template, "Chat template differs"\n            write_json(awq / "provenance.json", {"source_model": pin, "quantized_model": awq_pin,\n                "quantization_config": quantization, "same_architecture": True, "same_vocabulary": True,\n                "same_chat_template": True, "limitation": "Publisher identifies the source repo but does not pin its original quantization-source commit."})\n            awq_command = command.copy()\n            for flag in ("--model", "--tokenizer"):\n                awq_command[awq_command.index(flag)+1] = awq_repo\n            for flag in ("--revision", "--tokenizer-revision"):\n                awq_command[awq_command.index(flag)+1] = info.sha\n            awq_command += ["--quantization", "awq"]\n            server_smoke(awq, awq_command, settings, probe, {"tp": 1, "revision": info.sha, "quantization": "awq"})\n            record("awq", "VALIDATED", "Publisher-linked same-source AWQ, matching tokenizer/template, effective quantization=awq, FP16 activations, TP=1, real stream; no performance claim.", awq)\n        except BlockingIOError as exc:\n            record("awq", "NOT_TESTED", f"GPU prerequisite blocked; no AWQ execution: {exc}", awq)\n        except RequestException as exc:\n            record("awq", "UNAVAILABLE", f"Model metadata/download unavailable: {type(exc).__name__}: {exc}", awq)\n        except Exception as exc:\n            record("awq", "FAILED", f"{type(exc).__name__}: {exc}", awq)\n        write_json(q / "environment.json", environment)\n    except Exception as exc:\n        state["batch_error"] = f"{type(exc).__name__}: {exc}"\n        print("Batch stopped safely: " + state["batch_error"], flush=True)\n        for cap in CAPABILITIES[2:]:\n            if state[cap] == "NOT_TESTED":\n                state["reasons"][cap] = "Batch prerequisite failed: " + state["batch_error"]\n    finally:\n        state["qualification_complete"] = all(state[c] != "NOT_TESTED" for c in CAPABILITIES) and state["single_gpu_fp16_vllm"] == "VALIDATED"\n        state["limitations"] = ["Small correctness and generation smoke checks only; no SLO, p99, or performance certification.",\n            "Chunked prefill enabled failed in attempt b; baseline c explicitly disables it.",\n            "Kernel Lab portfolio implementation and all platform implementation remain unstarted.",\n            "Upstream cancellation/error-path coverage and exact runtime KV dtype introspection remain untested."]\n        write_json(q / "qualification.json", state)\n        if not (old / "qualification.json").exists():\n            failed = {"qualification_id": old.name, **{c: "NOT_TESTED" for c in CAPABILITIES},\n                "single_gpu_fp16_vllm": "FAILED", "streaming": "FAILED", "validated_versions": {}, "model": pin,\n                "qualification_complete": False, "failure": state["known_failure"],\n                "reason": "Engine crashed in chunked-prefill Triton kernel before any content; no completed stream or calculable metrics.",\n                "remaining_capabilities_reason": "Separate checks were not run under this failed attempt; see qualification c."}\n            write_json(old / "qualification.json", failed)\n        if not (old / "environment.json").exists() and (old / "environment-preflight.json").exists():\n            write_json(old / "environment.json", {**read_json(old / "environment-preflight.json"), "model": pin})\n        if not (old / "qualification-report.md").exists():\n            (old / "qualification-report.md").write_text("# Qualification " + old.name\n                + "\\n\\nFAILED: FP16 serving with chunked prefill enabled crashed on first generation. "\n                + "Triton prefix-prefill requested 98304 bytes shared memory; hardware limit was 65536. "\n                + "The client received no content and timed out. Empty stream-sample.txt represents no observed events. "\n                + "Versions were installed, not certified by this failed attempt. Remaining capabilities were NOT_TESTED. "\n                + "See preserved server-log.txt, diagnostics, and qualification.json. "\n                + "Follow-up c explicitly disables chunked prefill and keeps the same versions/model/dtype/backend.\\n")\n        rows = "\\n".join(f"| {cap} | {state[cap]} | {state[\'reasons\'][cap]} |" for cap in CAPABILITIES)\n        (q / "qualification-report.md").write_text("# Qualification " + q.name + "\\n\\n"\n            + "Baseline: pinned Qwen2.5-3B, one T4, FP16, TP=1, V0/XFORMERS, eager, prefix caching and chunked prefill disabled.\\n\\n"\n            + "| Capability | Status | Evidence / reason |\\n| --- | --- | --- |\\n" + rows\n            + "\\n\\nFull details, limitations, versions, and artifact references: `qualification.json`. "\n            + "The failed chunked-prefill attempt b remains preserved. No benchmark performance claim is made.\\n")\n        archive = q.parent.parent / ("inferencelab-qualification-" + batch.name + ".zip")\n        with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as out:\n            for directory in (old, q):\n                for current, dirs, files in os.walk(directory):\n                    dirs[:] = [d for d in dirs if d not in (".venv", "bootstrap", "__pycache__", ".cache")]\n                    for name in files:\n                        path = Path(current) / name\n                        if not path.is_symlink():\n                            out.write(path, path.relative_to(q.parent.parent))\n        print("\\n" + json.dumps({cap: state[cap] for cap in CAPABILITIES}, indent=2), flush=True)\n        print("Qualification complete:", state["qualification_complete"], flush=True)\n        print("DOWNLOAD THIS ARCHIVE:", archive, flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--directory", type=Path, default=Path("/kaggle/working/qualification/kaggle-t4-20261025-c"))\n    parser.add_argument("--self-check", action="store_true")\n    args = parser.parse_args()\n    if args.self_check:\n        self_check()\n    else:\n        main(args.directory)\n')
result = subprocess.run([sys.executable, '-u', str(script)])
if result.returncode:
    raise RuntimeError('Qualification did not finish; read the preserved logs above.')

from IPython.display import FileLink, display
archives = sorted(Path('/kaggle/working').glob('inferencelab-qualification-additional-*.zip'))
if archives:
    display(FileLink(str(archives[-1])))
